# 89. The Single vs. Dual Sourcing Problem

## Tier 4: The AI/ML/RL Augmentation Method (Deep Reinforcement Learning)

### Key assumptions
- Sourcing decisions can be formulated as a sequential decision-making problem
- Agent learns optimal policies through interaction with simulated environment
- State space captures demand forecasts, inventory levels, and supplier performance
- Action space includes sourcing strategy and allocation decisions
- Reward function balances cost efficiency, service levels, and supply chain resilience

### Approach (step-by-step)
1. **Environment Modeling**: Create sourcing simulation environment with state transitions
2. **State Space Definition**: Encode demand, inventory, reliability, and market conditions
3. **Action Space Design**: Define sourcing decisions as discrete/continuous actions
4. **Reward Function**: Multi-objective reward balancing competing priorities
5. **Neural Network Architecture**: Deep Q-Network for policy approximation
6. **Training Loop**: Experience replay and target network updates
7. **Policy Evaluation**: Assess learned policy performance and convergence

### What to look for in the results
- Learning curves showing reward improvement over episodes
- Policy convergence and stability analysis
- Comparison with baseline methods (heuristic, mathematical)
- Generalization to different demand scenarios
- Exploration vs. exploitation balance during training

### Concrete example (from the source)
We'll implement DQN with the MediTech scenario:
- 52-week simulation horizon with dynamic demand
- 3 components and 4 suppliers with time-varying performance
- State space: demand forecasts, inventory, supplier reliability trends
- Training: 1000 episodes, learning rate 0.001, replay buffer 10,000

In [1]:
# Import required libraries for Deep Reinforcement Learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Union
import random
import time
import warnings
from collections import deque, defaultdict
import copy
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Simple neural network implementation (no external dependencies)
class SimpleNeuralNetwork:
    """Simple feedforward neural network for DQN"""
    
    def __init__(self, input_size: int, hidden_size: int, output_size: int, learning_rate: float = 0.001):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        
        # Initialize weights and biases
        # Xavier initialization for better convergence
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        
    def relu(self, x: np.ndarray) -> np.ndarray:
        """ReLU activation function"""
        return np.maximum(0, x)
    
    def relu_derivative(self, x: np.ndarray) -> np.ndarray:
        """ReLU derivative for backpropagation"""
        return (x > 0).astype(float)
    
    def forward(self, X: np.ndarray) -> np.ndarray:
        """Forward propagation"""
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        return self.z2  # No activation in output layer for Q-values
    
    def backward(self, X: np.ndarray, y: np.ndarray, output: np.ndarray):
        """Backward propagation and weight update"""
        m = X.shape[0]
        
        # Output layer gradients
        dz2 = (output - y) / m
        dW2 = np.dot(self.a1.T, dz2)
        db2 = np.sum(dz2, axis=0, keepdims=True)
        
        # Hidden layer gradients
        dz1 = np.dot(dz2, self.W2.T) * self.relu_derivative(self.z1)
        dW1 = np.dot(X.T, dz1)
        db1 = np.sum(dz1, axis=0, keepdims=True)
        
        # Update weights
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Make predictions"""
        return self.forward(X)
    
    def copy(self):
        """Create a copy of the network"""
        new_net = SimpleNeuralNetwork(self.input_size, self.hidden_size, self.output_size, self.learning_rate)
        new_net.W1 = self.W1.copy()
        new_net.b1 = self.b1.copy()
        new_net.W2 = self.W2.copy()
        new_net.b2 = self.b2.copy()
        return new_net

In [ ]:
@dataclass
class RLComponent:
    """Component for reinforcement learning environment"""
    name: str
    criticality_score: float
    base_annual_demand: int
    demand_variance: float  # Demand variability
    lead_time: int  # Weeks
    holding_cost_rate: float  # Annual holding cost as % of unit cost

@dataclass
class RLSupplier:
    """Supplier for reinforcement learning environment"""
    name: str
    base_unit_cost: float
    base_reliability: float
    cost_volatility: float  # Cost variation over time
    reliability_trend: float  # Reliability trend (positive = improving)
    capacity_flexibility: float  # Ability to handle demand surges
    quality_score: float

@dataclass
class RLState:
    """State representation for RL environment"""
    week: int
    demand_forecasts: Dict[str, float]  # component -> demand forecast
    inventory_levels: Dict[str, float]  # component -> current inventory
    supplier_reliabilities: Dict[str, float]  # supplier -> current reliability
    supplier_costs: Dict[str, float]  # supplier -> current unit cost
    market_conditions: float  # Overall market condition (-1 to 1)
    disruption_risk: float  # Current disruption risk level

@dataclass
class RLAction:
    """Action representation for RL environment"""
    component_name: str
    sourcing_strategy: str  # 'single' or 'dual'
    primary_supplier: str
    secondary_supplier: Optional[str] = None
    allocation_ratio: float = 0.7  # For dual sourcing

In [ ]:
class SourcingEnvironment:
    """Reinforcement Learning environment for sourcing decisions"""
    
    def __init__(self, components: List[RLComponent], suppliers: List[RLSupplier], 
                 max_weeks: int = 52, service_level_target: float = 0.95):
        self.components = components
        self.suppliers = suppliers
        self.max_weeks = max_weeks
        self.service_level_target = service_level_target
        
        # Environment state
        self.current_week = 0
        self.state = None
        self.inventory = {comp.name: 0.0 for comp in components}
        self.total_cost = 0.0
        self.total_demand = 0.0
        self.total_satisfied_demand = 0.0
        
        # Market dynamics
        self.market_condition = 0.0  # -1 (bad) to 1 (good)
        self.disruption_risk = 0.1  # Base disruption probability
        
        # Action space definition
        self.action_space = self._create_action_space()
        self.state_size = self._calculate_state_size()
        
        # Performance tracking
        self.episode_cost_history = []
        self.episode_service_level_history = []
    
    def _create_action_space(self) -> List[RLAction]:
        """Create discrete action space for sourcing decisions"""
        actions = []
        
        for component in self.components:
            # Single sourcing actions
            for supplier in self.suppliers:
                actions.append(RLAction(
                    component_name=component.name,
                    sourcing_strategy='single',
                    primary_supplier=supplier.name
                ))
            
            # Dual sourcing actions (for high criticality components)
            if component.criticality_score > 0.7:
                for i, supplier1 in enumerate(self.suppliers):
                    for supplier2 in self.suppliers[i+1:]:
                        actions.append(RLAction(
                            component_name=component.name,
                            sourcing_strategy='dual',
                            primary_supplier=supplier1.name,
                            secondary_supplier=supplier2.name,
                            allocation_ratio=0.7
                        ))
        
        return actions
    
    def _calculate_state_size(self) -> int:
        """Calculate the size of the state representation"""
        # Week (1) + market conditions (2) + component data (3 per component) + supplier data (2 per supplier)
        return (1 + 2 + len(self.components) * 3 + len(self.suppliers) * 2)
    
    def reset(self) -> RLState:
        """Reset environment to initial state"""
        self.current_week = 0
        self.inventory = {comp.name: comp.lead_time * comp.base_annual_demand / 52 
                          for comp in self.components}  # Start with lead time inventory
        self.total_cost = 0.0
        self.total_demand = 0.0
        self.total_satisfied_demand = 0.0
        
        # Random initial market conditions
        self.market_condition = np.random.uniform(-0.3, 0.3)
        self.disruption_risk = 0.1 + max(0, -self.market_condition) * 0.2
        
        self.state = self._get_current_state()
        return self.state
    
    def _get_current_state(self) -> RLState:
        """Get current state representation"""
        demand_forecasts = {}
        for comp in self.components:
            # Demand forecast with market influence
            market_factor = 1.0 + self.market_condition * 0.2
            weekly_demand = comp.base_annual_demand / 52 * market_factor
            demand_forecasts[comp.name] = weekly_demand
        
        supplier_reliabilities = {}
        supplier_costs = {}
        for supplier in self.suppliers:
            # Dynamic reliability with trend and market influence
            reliability_trend_factor = 1.0 + supplier.reliability_trend * self.current_week / self.max_weeks
            market_impact = self.market_condition * 0.1
            current_reliability = supplier.base_reliability * reliability_trend_factor + market_impact
            supplier_reliabilities[supplier.name] = np.clip(current_reliability, 0.5, 0.99)
            
            # Dynamic cost with volatility and market influence
            cost_variation = 1.0 + np.sin(self.current_week / 10) * supplier.cost_volatility
            market_cost_impact = -self.market_condition * 0.15  # Good market = lower costs
            current_cost = supplier.base_unit_cost * cost_variation * (1 + market_cost_impact)
            supplier_costs[supplier.name] = max(1.0, current_cost)
        
        return RLState(
            week=self.current_week,
            demand_forecasts=demand_forecasts,
            inventory_levels=self.inventory.copy(),
            supplier_reliabilities=supplier_reliabilities,
            supplier_costs=supplier_costs,
            market_conditions=self.market_condition,
            disruption_risk=self.disruption_risk
        )
    
    def step(self, action: RLAction) -> Tuple[RLState, float, bool, Dict]:
        """Execute one step in the environment"""
        # Execute sourcing action
        cost, satisfied_demand = self._execute_action(action)
        
        # Update market conditions
        self._update_market_conditions()
        
        # Advance week
        self.current_week += 1
        
        # Calculate reward
        reward = self._calculate_reward(cost, satisfied_demand, action)
        
        # Check if episode is done
        done = self.current_week >= self.max_weeks
        
        # Get new state
        self.state = self._get_current_state()
        
        # Info dictionary
        info = {
            'cost': cost,
            'satisfied_demand': satisfied_demand,
            'service_level': self.total_satisfied_demand / max(1, self.total_demand),
            'total_cost': self.total_cost
        }
        
        return self.state, reward, done, info
    
    def _execute_action(self, action: RLAction) -> Tuple[float, float]:
        """Execute sourcing action and return cost and satisfied demand"""
        component = next(c for c in self.components if c.name == action.component_name)
        weekly_demand = self.state.demand_forecasts[component.name]
        
        total_cost = 0.0
        satisfied_demand = 0.0
        
        if action.sourcing_strategy == 'single':
            supplier = next(s for s in self.suppliers if s.name == action.primary_supplier)
            
            # Check for disruption
            if np.random.random() > self.state.supplier_reliabilities[supplier.name]:
                # Disruption occurred
                satisfied_demand = 0
                total_cost = weekly_demand * 50  # Emergency procurement cost
            else:
                # Normal delivery
                unit_cost = self.state.supplier_costs[supplier.name]
                total_cost = weekly_demand * unit_cost
                satisfied_demand = weekly_demand
                
        else:  # Dual sourcing
            supplier1 = next(s for s in self.suppliers if s.name == action.primary_supplier)
            supplier2 = next(s for s in self.suppliers if s.name == action.secondary_supplier)
            
            # Primary supplier
            primary_demand = weekly_demand * action.allocation_ratio
            if np.random.random() > self.state.supplier_reliabilities[supplier1.name]:
                # Primary disrupted, secondary takes full demand
                if np.random.random() > self.state.supplier_reliabilities[supplier2.name]:
                    # Both disrupted
                    satisfied_demand = 0
                    total_cost = weekly_demand * 50
                else:
                    # Only secondary delivers
                    unit_cost = self.state.supplier_costs[supplier2.name]
                    total_cost = weekly_demand * unit_cost * 1.2  # Premium for rush
                    satisfied_demand = weekly_demand * 0.9  # Some loss due to rush
            else:
                # Primary delivers
                unit_cost1 = self.state.supplier_costs[supplier1.name]
                total_cost += primary_demand * unit_cost1
                satisfied_demand += primary_demand
                
                # Secondary supplier (backup)
                secondary_demand = weekly_demand * (1 - action.allocation_ratio)
                if np.random.random() > self.state.supplier_reliabilities[supplier2.name]:
                    # Secondary disrupted
                    satisfied_demand += secondary_demand * 0.5  # Partial from inventory
                    total_cost += secondary_demand * unit_cost1 * 0.3  # Holding cost
                else:
                    unit_cost2 = self.state.supplier_costs[supplier2.name]
                    total_cost += secondary_demand * unit_cost2
                    satisfied_demand += secondary_demand
        
        # Update totals
        self.total_cost += total_cost
        self.total_demand += weekly_demand
        self.total_satisfied_demand += satisfied_demand
        
        return total_cost, satisfied_demand
    
    def _update_market_conditions(self):
        """Update market conditions for next period"""
        # Random walk with mean reversion
        self.market_condition += np.random.normal(0, 0.1)
        self.market_condition *= 0.95  # Mean reversion toward 0
        self.market_condition = np.clip(self.market_condition, -0.8, 0.8)
        
        # Update disruption risk
        self.disruption_risk = 0.1 + max(0, -self.market_condition) * 0.2
        self.disruption_risk += np.random.normal(0, 0.02)
        self.disruption_risk = np.clip(self.disruption_risk, 0.05, 0.5)
    
    def _calculate_reward(self, cost: float, satisfied_demand: float, action: RLAction) -> float:
        """Calculate reward for the action"""
        weekly_demand = self.state.demand_forecasts[action.component_name]
        
        # Cost efficiency reward (lower cost = higher reward)
        cost_efficiency = -cost / weekly_demand
        
        # Service level reward
        service_level = satisfied_demand / weekly_demand
        service_reward = service_level * 10  # Scale to make it significant
        
        # Risk mitigation reward for dual sourcing (when critical)
        component = next(c for c in self.components if c.name == action.component_name)
        risk_reward = 0
        if action.sourcing_strategy == 'dual' and component.criticality_score > 0.7:
            risk_reward = component.criticality_score * 2
        
        # Total reward
        total_reward = cost_efficiency + service_reward + risk_reward
        
        return total_reward
    
    def state_to_vector(self, state: RLState) -> np.ndarray:
        """Convert state to vector representation for neural network"""
        vector = []
        
        # Time features
        vector.append(state.week / self.max_weeks)  # Normalized week
        vector.append(state.market_conditions)
        vector.append(state.disruption_risk)
        
        # Component features
        for comp in self.components:
            vector.append(state.demand_forecasts[comp.name] / 1000)  # Normalized demand
            vector.append(state.inventory_levels[comp.name] / 1000)  # Normalized inventory
            vector.append(comp.criticality_score)
        
        # Supplier features
        for supplier in self.suppliers:
            vector.append(state.supplier_reliabilities[supplier.name])
            vector.append(state.supplier_costs[supplier.name] / 20)  # Normalized cost
        
        return np.array(vector, dtype=np.float32)

In [ ]:
class DeepQLearningAgent:
    """Deep Q-Learning agent for sourcing optimization"""
    
    def __init__(self, state_size: int, action_size: int, hidden_size: int = 128,
                 learning_rate: float = 0.001, discount_factor: float = 0.95,
                 epsilon_start: float = 1.0, epsilon_end: float = 0.01, epsilon_decay: float = 0.995):
        """
        Initialize DQN agent
        
        Args:
            state_size: Dimension of state space
            action_size: Number of possible actions
            hidden_size: Hidden layer size in neural network
            learning_rate: Learning rate for neural network
            discount_factor: Discount factor for future rewards
            epsilon_start: Initial exploration rate
            epsilon_end: Final exploration rate
            epsilon_decay: Rate of epsilon decay
        """
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.gamma = discount_factor
        
        # Exploration parameters
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Neural networks
        self.q_network = SimpleNeuralNetwork(state_size, hidden_size, action_size, learning_rate)
        self.target_network = SimpleNeuralNetwork(state_size, hidden_size, action_size, learning_rate)
        self.update_target_network()
        
        # Experience replay
        self.memory = deque(maxlen=10000)
        self.batch_size = 32
        
        # Training metrics
        self.training_loss = []
        self.epsilon_history = []
        self.q_value_history = []
    
    def update_target_network(self):
        """Copy weights from main network to target network"""
        self.target_network = self.q_network.copy()
    
    def remember(self, state: np.ndarray, action: int, reward: float, next_state: np.ndarray, done: bool):
        """Store experience in replay memory"""
        self.memory.append((state, action, reward, next_state, done))
    
    def act(self, state: np.ndarray) -> int:
        """Choose action using epsilon-greedy policy"""
        if np.random.random() <= self.epsilon:
            return random.randrange(self.action_size)
        
        # Exploit: choose best action from Q-values
        state_reshaped = state.reshape(1, -1)
        q_values = self.q_network.predict(state_reshaped)
        
        # Store Q-value for tracking
        self.q_value_history.append(np.max(q_values))
        
        return np.argmax(q_values[0])
    
    def replay(self):
        """Train the model on a batch of experiences"""
        if len(self.memory) < self.batch_size:
            return
        
        # Sample random batch from memory
        batch = random.sample(self.memory, self.batch_size)
        
        # Prepare batch data
        states = np.array([e[0] for e in batch])
        actions = np.array([e[1] for e in batch])
        rewards = np.array([e[2] for e in batch])
        next_states = np.array([e[3] for e in batch])
        dones = np.array([e[4] for e in batch])
        
        # Get current Q-values
        current_q_values = self.q_network.predict(states)
        
        # Get next Q-values from target network
        next_q_values = self.target_network.predict(next_states)
        max_next_q_values = np.max(next_q_values, axis=1)
        
        # Calculate target Q-values
        target_q_values = current_q_values.copy()
        for i in range(self.batch_size):
            if dones[i]:
                target_q_values[i, actions[i]] = rewards[i]
            else:
                target_q_values[i, actions[i]] = rewards[i] + self.gamma * max_next_q_values[i]
        
        # Train the network
        self.q_network.backward(states, target_q_values, current_q_values)
        
        # Calculate and store loss
        loss = np.mean(np.square(target_q_values - current_q_values))
        self.training_loss.append(loss)
    
    def train(self, env: SourcingEnvironment, n_episodes: int = 1000):
        """
        Train the agent in the environment
        
        Args:
            env: Sourcing environment
            n_episodes: Number of training episodes
        """
        episode_rewards = []
        episode_costs = []
        episode_service_levels = []
        
        print(f"Starting DQN training for {n_episodes} episodes...")
        
        for episode in range(n_episodes):
            state = env.reset()
            state_vector = env.state_to_vector(state)
            total_reward = 0
            
            for week in range(env.max_weeks):
                # Choose action
                action_index = self.act(state_vector)
                action = env.action_space[action_index]
                
                # Take action
                next_state, reward, done, info = env.step(action)
                next_state_vector = env.state_to_vector(next_state)
                
                # Store experience
                self.remember(state_vector, action_index, reward, next_state_vector, done)
                
                # Update state
                state = next_state
                state_vector = next_state_vector
                total_reward += reward
                
                if done:
                    break
            
            # Train the agent
            self.replay()
            
            # Update epsilon
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
            self.epsilon_history.append(self.epsilon)
            
            # Update target network periodically
            if episode % 100 == 0:
                self.update_target_network()
            
            # Record episode metrics
            episode_rewards.append(total_reward)
            episode_costs.append(info['total_cost'])
            episode_service_levels.append(info['service_level'])
            
            # Progress reporting
            if (episode + 1) % 100 == 0:
                avg_reward = np.mean(episode_rewards[-100:])
                avg_cost = np.mean(episode_costs[-100:])
                avg_service = np.mean(episode_service_levels[-100:])
                print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.2f}, "
                      f"Avg Cost = ${avg_cost:.0f}, Avg Service = {avg_service:.3f}, "
                      f"Epsilon = {self.epsilon:.3f}")
        
        print(f"Training completed!")
        
        return {
            'episode_rewards': episode_rewards,
            'episode_costs': episode_costs,
            'episode_service_levels': episode_service_levels,
            'training_loss': self.training_loss,
            'epsilon_history': self.epsilon_history,
            'q_value_history': self.q_value_history
        }
    
    def evaluate(self, env: SourcingEnvironment, n_episodes: int = 100) -> Dict:
        """
        Evaluate the trained agent
        
        Args:
            env: Sourcing environment
            n_episodes: Number of evaluation episodes
        
        Returns:
            Dictionary with evaluation metrics
        """
        print(f"Evaluating agent for {n_episodes} episodes...")
        
        # Set epsilon to 0 for evaluation (no exploration)
        original_epsilon = self.epsilon
        self.epsilon = 0
        
        evaluation_rewards = []
        evaluation_costs = []
        evaluation_service_levels = []
        action_counts = defaultdict(int)
        
        for episode in range(n_episodes):
            state = env.reset()
            state_vector = env.state_to_vector(state)
            total_reward = 0
            
            for week in range(env.max_weeks):
                # Choose action (no exploration)
                action_index = self.act(state_vector)
                action = env.action_space[action_index]
                action_counts[action_index] += 1
                
                # Take action
                next_state, reward, done, info = env.step(action)
                next_state_vector = env.state_to_vector(next_state)
                
                # Update state
                state = next_state
                state_vector = next_state_vector
                total_reward += reward
                
                if done:
                    break
            
            # Record episode metrics
            evaluation_rewards.append(total_reward)
            evaluation_costs.append(info['total_cost'])
            evaluation_service_levels.append(info['service_level'])
        
        # Restore original epsilon
        self.epsilon = original_epsilon
        
        # Analyze action preferences
        most_common_actions = sorted(action_counts.items(), key=lambda x: x[1], reverse=True)[:10]
        
        results = {
            'avg_reward': np.mean(evaluation_rewards),
            'avg_cost': np.mean(evaluation_costs),
            'avg_service_level': np.mean(evaluation_service_levels),
            'std_cost': np.std(evaluation_costs),
            'std_service_level': np.std(evaluation_service_levels),
            'min_cost': np.min(evaluation_costs),
            'max_cost': np.max(evaluation_costs),
            'min_service_level': np.min(evaluation_service_levels),
            'max_service_level': np.max(evaluation_service_levels),
            'most_common_actions': most_common_actions
        }
        
        print(f"Evaluation Results:")
        print(f"  Average Cost: ${results['avg_cost']:,.0f} ± ${results['std_cost']:,.0f}")
        print(f"  Average Service Level: {results['avg_service_level']:.3f} ± {results['std_service_level']:.3f}")
        print(f"  Cost Range: ${results['min_cost']:,.0f} - ${results['max_cost']:,.0f}")
        print(f"  Service Level Range: {results['min_service_level']:.3f} - {results['max_service_level']:.3f}")
        
        return results

In [ ]:
# Create the RL example environment
def create_rl_environment():
    """Create the sourcing environment for RL training"""
    
    # 3 components with varying characteristics
    components = [
        RLComponent(
            name="Component_1_Critical",
            criticality_score=0.9,
            base_annual_demand=460,
            demand_variance=0.2,
            lead_time=4,
            holding_cost_rate=0.25
        ),
        RLComponent(
            name="Component_2_Standard",
            criticality_score=0.6,
            base_annual_demand=810,
            demand_variance=0.15,
            lead_time=2,
            holding_cost_rate=0.20
        ),
        RLComponent(
            name="Component_3_Medium",
            criticality_score=0.75,
            base_annual_demand=320,
            demand_variance=0.25,
            lead_time=3,
            holding_cost_rate=0.22
        )
    ]
    
    # 4 suppliers with dynamic characteristics
    suppliers = [
        RLSupplier(
            name="Supplier_A",
            base_unit_cost=15,
            base_reliability=0.85,
            cost_volatility=0.1,
            reliability_trend=0.02,  # Improving reliability
            capacity_flexibility=0.8,
            quality_score=0.87
        ),
        RLSupplier(
            name="Supplier_B",
            base_unit_cost=18,
            base_reliability=0.92,
            cost_volatility=0.08,
            reliability_trend=-0.01,  # Slightly declining
            capacity_flexibility=0.9,
            quality_score=0.94
        ),
        RLSupplier(
            name="Supplier_C",
            base_unit_cost=16,
            base_reliability=0.88,
            cost_volatility=0.12,
            reliability_trend=0.01,
            capacity_flexibility=0.7,
            quality_score=0.90
        ),
        RLSupplier(
            name="Supplier_D",
            base_unit_cost=14,
            base_reliability=0.82,
            cost_volatility=0.15,
            reliability_trend=0.03,
            capacity_flexibility=0.6,
            quality_score=0.85
        )
    ]
    
    return SourcingEnvironment(components, suppliers, max_weeks=52)

# Create environment
env = create_rl_environment()
print(f"Environment created with {len(env.components)} components and {len(env.suppliers)} suppliers")
print(f"Action space size: {len(env.action_space)} possible actions")
print(f"State size: {env.state_size} dimensions")

In [ ]:
# Initialize and train the DQN agent
agent = DeepQLearningAgent(
    state_size=env.state_size,
    action_size=len(env.action_space),
    hidden_size=128,
    learning_rate=0.001,
    discount_factor=0.95,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995
)

# Train the agent
training_results = agent.train(env, n_episodes=1000)

In [ ]:
# Evaluate the trained agent
evaluation_results = agent.evaluate(env, n_episodes=100)

In [ ]:
# Analyze learned policy and training behavior
def analyze_training_results():
    """Analyze training results and learned policy"""
    
    print("=== TRAINING ANALYSIS ===")
    
    # Learning curves
    episode_rewards = training_results['episode_rewards']
    episode_costs = training_results['episode_costs']
    episode_service_levels = training_results['episode_service_levels']
    
    # Calculate moving averages for smoother plots
    window_size = 50
    rewards_ma = pd.Series(episode_rewards).rolling(window=window_size).mean()
    costs_ma = pd.Series(episode_costs).rolling(window=window_size).mean()
    service_ma = pd.Series(episode_service_levels).rolling(window=window_size).mean()
    
    # Find convergence point
    convergence_episode = None
    for i in range(window_size, len(rewards_ma)):
        recent_avg = rewards_ma.iloc[i]
        previous_avg = rewards_ma.iloc[i-window_size]
        if abs(recent_avg - previous_avg) / abs(previous_avg) < 0.01:  # 1% improvement threshold
            convergence_episode = i
            break
    
    if convergence_episode:
        print(f"Policy convergence around episode {convergence_episode}")
        print(f"Final average reward: {rewards_ma.iloc[-1]:.2f}")
    else:
        print(f"Policy still improving after {len(episode_rewards)} episodes")
    
    # Policy analysis
    print("\n=== LEARNED POLICY ANALYSIS ===")
    most_common_actions = evaluation_results['most_common_actions']
    
    print("Most common actions:")
    for action_idx, count in most_common_actions[:10]:
        action = env.action_space[action_idx]
        percentage = (count / (100 * 52)) * 100  # 100 episodes * 52 weeks
        print(f"  {action.component_name}: {action.sourcing_strategy} sourcing")
        if action.sourcing_strategy == 'single':
            print(f"    From {action.primary_supplier} ({percentage:.1f}% of decisions)")
        else:
            print(f"    {action.allocation_ratio*100:.0f}% {action.primary_supplier}, "
                  f"{(1-action.allocation_ratio)*100:.0f}% {action.secondary_supplier} ({percentage:.1f}%)")
    
    return {
        'rewards_ma': rewards_ma,
        'costs_ma': costs_ma,
        'service_ma': service_ma,
        'convergence_episode': convergence_episode
    }

analysis_results = analyze_training_results()

In [ ]:
# Compare with baseline policies
def compare_with_baselines():
    """Compare DQN policy with simple baseline policies"""
    
    print("=== BASELINE COMPARISON ===")
    
    # Baseline 1: Always single source from cheapest supplier
    def cheapest_policy(state: RLState) -> RLAction:
        # Find cheapest supplier for highest demand component
        highest_demand_comp = max(state.demand_forecasts.items(), key=lambda x: x[1])[0]
        cheapest_supplier = min(state.supplier_costs.items(), key=lambda x: x[1])[0]
        
        return RLAction(
            component_name=highest_demand_comp,
            sourcing_strategy='single',
            primary_supplier=cheapest_supplier
        )
    
    # Baseline 2: Always dual source for critical components
    def dual_critical_policy(state: RLState) -> RLAction:
        # Find most critical component
        critical_comp = max(env.components, key=lambda c: c.criticality_score).name
        
        # Get two most reliable suppliers
        reliable_suppliers = sorted(state.supplier_reliabilities.items(), 
                                  key=lambda x: x[1], reverse=True)[:2]
        
        return RLAction(
            component_name=critical_comp,
            sourcing_strategy='dual',
            primary_supplier=reliable_suppliers[0][0],
            secondary_supplier=reliable_suppliers[1][0],
            allocation_ratio=0.7
        )
    
    # Evaluate baselines
    def evaluate_policy(policy_func, name: str, n_episodes: int = 50):
        print(f"\nEvaluating {name} policy...")
        
        total_costs = []
        total_service_levels = []
        
        for episode in range(n_episodes):
            state = env.reset()
            episode_cost = 0
            episode_demand = 0
            episode_satisfied = 0
            
            for week in range(env.max_weeks):
                action = policy_func(state)
                next_state, reward, done, info = env.step(action)
                
                episode_cost += info['cost']
                episode_demand += sum(state.demand_forecasts.values())
                episode_satisfied += info['satisfied_demand']
                
                state = next_state
                
                if done:
                    break
            
            total_costs.append(episode_cost)
            service_level = episode_satisfied / max(1, episode_demand)
            total_service_levels.append(service_level)
        
        avg_cost = np.mean(total_costs)
        avg_service = np.mean(total_service_levels)
        
        print(f"  Average Cost: ${avg_cost:,.0f}")
        print(f"  Average Service Level: {avg_service:.3f}")
        
        return avg_cost, avg_service
    
    # Evaluate baselines
    cheapest_cost, cheapest_service = evaluate_policy(cheapest_policy, "Cheapest Single Sourcing")
    dual_cost, dual_service = evaluate_policy(dual_critical_policy, "Dual Critical Sourcing")
    
    # Comparison table
    comparison_data = [
        {
            'Method': 'DQN (Learned)',
            'Avg_Cost': evaluation_results['avg_cost'],
            'Service_Level': evaluation_results['avg_service_level'],
            'Cost_Std': evaluation_results['std_cost'],
            'Service_Std': evaluation_results['std_service_level']
        },
        {
            'Method': 'Cheapest Single',
            'Avg_Cost': cheapest_cost,
            'Service_Level': cheapest_service,
            'Cost_Std': np.std([cheapest_cost] * 50),  # No variation in deterministic policy
            'Service_Std': np.std([cheapest_service] * 50)
        },
        {
            'Method': 'Dual Critical',
            'Avg_Cost': dual_cost,
            'Service_Level': dual_service,
            'Cost_Std': np.std([dual_cost] * 50),
            'Service_Std': np.std([dual_service] * 50)
        }
    ]
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + comparison_df.to_string(index=False))
    
    # Calculate improvements
    dqn_cost = evaluation_results['avg_cost']
    dqn_service = evaluation_results['avg_service_level']
    
    cost_improvement_vs_cheapest = ((cheapest_cost - dqn_cost) / cheapest_cost) * 100
    service_improvement_vs_cheapest = ((dqn_service - cheapest_service) / cheapest_service) * 100
    
    cost_improvement_vs_dual = ((dual_cost - dqn_cost) / dual_cost) * 100
    service_improvement_vs_dual = ((dqn_service - dual_service) / dual_service) * 100
    
    print(f"\nDQN vs. Cheapest Single:")
    print(f"  Cost improvement: {cost_improvement_vs_cheapest:+.1f}%")
    print(f"  Service improvement: {service_improvement_vs_cheapest:+.1f}%")
    
    print(f"\nDQN vs. Dual Critical:")
    print(f"  Cost improvement: {cost_improvement_vs_dual:+.1f}%")
    print(f"  Service improvement: {service_improvement_vs_dual:+.1f}%")
    
    return comparison_df

comparison_df = compare_with_baselines()

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Deep Reinforcement Learning Analysis', fontsize=16, fontweight='bold')

# 1. Learning Curves
ax1 = axes[0, 0]
episodes = range(len(training_results['episode_rewards']))
ax1.plot(episodes, training_results['episode_rewards'], 'b-', alpha=0.3, label='Episode Reward')
ax1.plot(episodes, analysis_results['rewards_ma'], 'b-', linewidth=2, label='Moving Average (50)')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.set_title('Learning Curve - Reward')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Mark convergence point
if analysis_results['convergence_episode']:
    ax1.axvline(x=analysis_results['convergence_episode'], color='red', linestyle='--', 
               alpha=0.7, label=f'Convergence (~Ep {analysis_results['convergence_episode']})')
    ax1.legend()

# 2. Cost and Service Level Progress
ax2 = axes[0, 1]
ax2_cost = ax2.twinx()

# Plot cost (left axis)
line1 = ax2.plot(episodes, training_results['episode_costs'], 'r-', alpha=0.3, label='Episode Cost')
line2 = ax2.plot(episodes, analysis_results['costs_ma'], 'r-', linewidth=2, label='Avg Cost (50)')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Total Cost ($)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Plot service level (right axis)
line3 = ax2_cost.plot(episodes, training_results['episode_service_levels'], 'g-', alpha=0.3, label='Episode Service')
line4 = ax2_cost.plot(episodes, analysis_results['service_ma'], 'g-', linewidth=2, label='Avg Service (50)')
ax2_cost.set_ylabel('Service Level', color='green')
ax2_cost.tick_params(axis='y', labelcolor='green')

ax2.set_title('Learning Progress - Cost & Service')
lines = line1 + line2 + line3 + line4
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc='upper right')
ax2.grid(True, alpha=0.3)

# 3. Epsilon Decay
ax3 = axes[1, 0]
ax3.plot(range(len(training_results['epsilon_history'])), 
         training_results['epsilon_history'], 'purple', linewidth=2)
ax3.set_xlabel('Episode')
ax3.set_ylabel('Epsilon (Exploration Rate)')
ax3.set_title('Exploration Rate Decay')
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, 1.1)

# Add annotations
ax3.axhline(y=0.01, color='red', linestyle='--', alpha=0.7, label='Min Epsilon')
ax3.legend()

# 4. Method Comparison
ax4 = axes[1, 1]
x_pos = np.arange(len(comparison_df))
width = 0.35

# Cost comparison
bars1 = ax4.bar(x_pos - width/2, comparison_df['Avg_Cost'], width, 
                label='Average Cost', color='lightcoral', alpha=0.8)

# Service level comparison (scaled for visibility)
scaled_service = comparison_df['Service_Level'] * 10000  # Scale to cost range
bars2 = ax4.bar(x_pos + width/2, scaled_service, width, 
                label='Service Level (×10000)', color='lightblue', alpha=0.8)

ax4.set_xlabel('Method')
ax4.set_ylabel('Value')
ax4.set_title('Policy Comparison')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(comparison_df['Method'], rotation=45)
ax4.legend()

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax4.text(bar.get_x() + bar.get_width()/2., height + 100,
                    f'{height:.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Detailed policy analysis and insights
def analyze_learned_policy_insights():
    """Analyze insights from the learned policy"""
    
    print("=== POLICY INSIGHTS ===")
    
    # Analyze action preferences by component criticality
    component_preferences = defaultdict(lambda: defaultdict(int))
    
    for action_idx, count in evaluation_results['most_common_actions']:
        action = env.action_space[action_idx]
        component = action.component_name
        strategy = action.sourcing_strategy
        component_preferences[component][strategy] += count
    
    print("\nSourcing Strategy Preferences by Component:")
    for component in env.components:
        comp_name = component.name
        preferences = component_preferences[comp_name]
        total = sum(preferences.values())
        
        print(f"\n{comp_name} (Criticality: {component.criticality_score:.2f}):")
        for strategy, count in preferences.items():
            percentage = (count / total) * 100 if total > 0 else 0
            print(f"  {strategy.title()} sourcing: {percentage:.1f}% ({count} decisions)")
    
    # Analyze supplier preferences
    supplier_preferences = defaultdict(int)
    
    for action_idx, count in evaluation_results['most_common_actions']:
        action = env.action_space[action_idx]
        supplier_preferences[action.primary_supplier] += count
        if action.secondary_supplier:
            supplier_preferences[action.secondary_supplier] += count * 0.3  # Weight for secondary
    
    print("\n=== SUPPLIER PREFERENCES ===")
    total_supplier_usage = sum(supplier_preferences.values())
    for supplier in env.suppliers:
        usage = supplier_preferences[supplier.name]
        percentage = (usage / total_supplier_usage) * 100 if total_supplier_usage > 0 else 0
        print(f"{supplier.name}: {percentage:.1f}% usage")
    
    # Analyze adaptation to market conditions
    print("\n=== MARKET CONDITION ADAPTATION ===")
    
    # Test policy under different market conditions
    market_scenarios = [
        ("Good Market", 0.5),
        ("Normal Market", 0.0),
        ("Bad Market", -0.5)
    ]
    
    for scenario_name, market_condition in market_scenarios:
        # Set market condition
        env.market_condition = market_condition
        
        # Get state and action
        state = env._get_current_state()
        state_vector = env.state_to_vector(state)
        
        # Get best action from learned policy
        q_values = agent.q_network.predict(state_vector.reshape(1, -1))
        best_action_idx = np.argmax(q_values[0])
        best_action = env.action_space[best_action_idx]
        
        print(f"\n{scenario_name} (Market: {market_condition:+.1f}):")
        print(f"  Recommended action: {best_action.component_name}")
        print(f"  Strategy: {best_action.sourcing_strategy} sourcing")
        if best_action.sourcing_strategy == 'single':
            print(f"  From: {best_action.primary_supplier}")
        else:
            print(f"  {best_action.allocation_ratio*100:.0f}% {best_action.primary_supplier}, "
                  f"{(1-best_action.allocation_ratio)*100:.0f}% {best_action.secondary_supplier}")
        
        print(f"  Q-value: {q_values[0][best_action_idx]:.2f}")

analyze_learned_policy_insights()

### Why this Tier exists vs earlier Tiers
This Tier 4 provides advanced AI/ML augmentation that addresses limitations of traditional optimization:
- **Adaptive learning**: Agent learns optimal policies from experience, not predefined rules
- **Dynamic environment handling**: Adapts to changing market conditions and supplier performance
- **Sequential decision making**: Captures temporal dependencies and long-term consequences
- **Experience-based optimization**: Improves performance through interaction with environment
- **Policy generalization**: Learned policy can handle unseen scenarios and variations

### Pros / Cons vs earlier Tiers
**Advantages over Tier 1 (Mathematical Optimization):**
- **Dynamic adaptation**: Responds to real-time changes vs. static optimization
- **Scalability**: Handles complex, high-dimensional state spaces efficiently
- **Learning from data**: Improves with more experience vs. fixed mathematical models
- **Uncertainty handling**: Naturally incorporates stochastic elements and risk
- **Policy speed**: Fast decision making after training vs. real-time optimization

**Advantages over Tier 2 (Priority Heuristic):**
- **Optimized policies**: Learns optimal decision rules vs. handcrafted heuristics
- **Context awareness**: Considers full state space vs. limited heuristic factors
- **Adaptive behavior**: Adjusts to changing conditions vs. fixed rules
- **Long-term optimization**: Balances immediate vs. future rewards vs. myopic decisions
- **Discovery of patterns**: Can discover non-obvious strategies vs. human-designed rules

**Advantages over Tier 3 (ACO):**
- **Temporal learning**: Captures time-dependent patterns vs. static optimization
- **Policy representation**: Compact neural network policy vs. population of solutions
- **Online adaptation**: Can continue learning in real-time vs. batch optimization
- **State complexity**: Handles rich state representations vs. limited problem encoding
- **Transferability**: Learned policy can be applied to similar problems

**Disadvantages:**
- **Training complexity**: Requires significant computational resources for training
- **Hyperparameter sensitivity**: Performance depends on neural network and RL parameters
- **Data requirements**: Needs sufficient training experience to learn good policies
- **Black box nature**: Learned policies can be difficult to interpret and explain
- **Convergence uncertainty**: No guarantee of finding optimal policy

### When to use this Tier
- **Dynamic environments**: When market conditions and supplier performance change frequently
- **Sequential decisions**: When decisions have long-term consequences and temporal dependencies
- **Large-scale problems**: When state space is too complex for traditional methods
- **Adaptive requirements**: When system needs to continuously improve from experience
- **Real-time decisions**: When fast decision making is critical after initial training
- **Uncertain conditions**: When operating under high uncertainty and stochastic elements
- **Complex constraints**: When problem has many interacting variables and constraints